# Fine-tune DGS để cắt câu ASL liên tục

Notebook này fine-tune checkpoint Moryossef DGS để tìm ranh giới câu từ pose MediaPipe Holistic 3D. Đây là **sentence segmentation**, không phải model dịch ASL.

## Cần chuẩn bị trong Google Drive

Đặt một tập con của video liên tục frontal và annotation How2Sign theo cấu trúc:

- how2sign_asl_boundary/train/how2sign_realigned_train.csv
- how2sign_asl_boundary/train/videos/<VIDEO_NAME>.mp4
- how2sign_asl_boundary/validation/how2sign_realigned_val.csv
- how2sign_asl_boundary/validation/videos/<VIDEO_NAME>.mp4

CSV cần các cột VIDEO_NAME, START_REALIGNED, END_REALIGNED; thời gian tính bằng giây. Dùng video liên tục, không dùng các clip đã cắt sẵn theo từng câu. Khởi đầu nên có khoảng 20 video train và 5 validation, đa dạng người ký/chủ đề. Trang How2Sign công bố train video khoảng 290 GB và validation 16 GB; notebook không tải toàn bộ dataset. [How2Sign chính thức](https://how2sign.github.io/)

Không dùng 5 video test đã đánh giá trong pilot để train/validation. Giữ test riêng. Timestamp dịch tiếng Anh có thể lệch ranh giới ASL nên validation chỉ là chỉ báo ban đầu.

Máy local có RTX 5060 8 GB VRAM, nhưng .venv-p4 đang dùng PyTorch CPU-only. Fine-tune local có thể thực hiện sau khi dựng môi trường CUDA riêng; notebook này dùng Colab GPU.

In [ ]:
%pip install -q "git+https://github.com/sign-language-processing/segmentation.git@v2026.4.27" "mediapipe==0.10.21"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, random, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Chọn Runtime > Change runtime type > GPU rồi chạy lại.")
DEVICE = torch.device("cuda")
print(f"GPU: {torch.cuda.get_device_name(0)} · PyTorch: {torch.__version__} · VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

ROOT = Path("/content/drive/MyDrive/how2sign_asl_boundary")
SPLITS = {
    "train": (ROOT / "train" / "how2sign_realigned_train.csv", ROOT / "train" / "videos"),
    "validation": (ROOT / "validation" / "how2sign_realigned_val.csv", ROOT / "validation" / "videos"),
}
POSE_ROOT = ROOT / "pose_cache"
FEATURE_ROOT = ROOT / "feature_cache"
OUT = ROOT / "checkpoints" / "dgs_how2sign_asl_finetuned"
for path in (POSE_ROOT, FEATURE_ROOT, OUT):
    path.mkdir(parents=True, exist_ok=True)

from sign_language_segmentation.bin import load_model, resolve_model_path
from sign_language_segmentation.utils.pose import preprocess_pose, compute_velocity
from sign_language_segmentation.utils.bio import BIO, create_bio_from_times
from sign_language_segmentation.metrics import likeliest_probs_to_segments, filter_segments
from pose_format import Pose

BASE = Path(resolve_model_path())
assert (BASE / "model.safetensors").is_file() and (BASE / "config.json").is_file(), f"Thiếu checkpoint/config: {BASE}"
print(f"Checkpoint khởi tạo: {BASE}")

In [ ]:
def load_split(split):
    annotation_path, video_dir = SPLITS[split]
    if not annotation_path.is_file() or not video_dir.is_dir():
        raise FileNotFoundError(f"Thiếu CSV hoặc thư mục video của split {split}: {annotation_path}, {video_dir}")
    frame = pd.read_csv(annotation_path, sep=None, engine="python")
    frame.columns = [str(c).strip().upper() for c in frame.columns]
    required = {"VIDEO_NAME", "START_REALIGNED", "END_REALIGNED"}
    if not required.issubset(frame.columns):
        raise ValueError(f"CSV phải có các cột {sorted(required)}")
    frame["VIDEO_NAME"] = frame["VIDEO_NAME"].astype(str).str.strip()
    frame["START_REALIGNED"] = pd.to_numeric(frame["START_REALIGNED"], errors="coerce")
    frame["END_REALIGNED"] = pd.to_numeric(frame["END_REALIGNED"], errors="coerce")
    frame = frame.dropna(subset=["VIDEO_NAME", "START_REALIGNED", "END_REALIGNED"])
    frame = frame[frame["END_REALIGNED"] > frame["START_REALIGNED"]]
    available = {p.stem for p in video_dir.glob("*.mp4")}
    selected = set(frame["VIDEO_NAME"]) & available
    if not selected:
        raise ValueError(f"Không có VIDEO_NAME nào khớp MP4 trong {video_dir}")
    missing = set(frame["VIDEO_NAME"]) - available
    frame = frame[frame["VIDEO_NAME"].isin(selected)].copy()
    print(f"{split}: {len(selected)} video, {len(frame)} câu; bỏ qua {len(missing)} video chưa có MP4.")
    return frame

train_df = load_split("train")
val_df = load_split("validation")
train_ids = set(train_df.VIDEO_NAME)
val_ids = set(val_df.VIDEO_NAME)
if train_ids & val_ids:
    raise ValueError("Có video trùng giữa train và validation.")
if len(train_ids) < 5 or len(val_ids) < 2:
    raise ValueError("Cần tối thiểu 5 train và 2 validation video; khuyến nghị 20/5 trở lên.")

In [ ]:
import cv2
import mediapipe as mp
from pose_anonymization.data import load_mean_and_std_pose
from pose_format.numpy import NumPyPoseBody
from pose_format.pose_header import PoseHeader, PoseHeaderDimensions
from pose_format.utils.holistic import body_points, component_points

_TEMPLATE, _ = load_mean_and_std_pose()
_COMPONENTS = {c.name: c for c in _TEMPLATE.header.components}
_COMPONENTS = [_COMPONENTS[n] for n in ("POSE_LANDMARKS", "LEFT_HAND_LANDMARKS", "RIGHT_HAND_LANDMARKS")]

def make_pose(video_path, pose_path):
    if pose_path.is_file() and pose_path.stat().st_size:
        return
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Không mở được video: {video_path}")
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if fps <= 0 or width <= 0 or height <= 0:
        cap.release()
        raise RuntimeError(f"Metadata video không hợp lệ: {video_path}")
    header = PoseHeader(version=_TEMPLATE.header.version, dimensions=PoseHeaderDimensions(width=width, height=height, depth=0), components=_COMPONENTS)
    data, confidence = [], []
    try:
        with mp.solutions.holistic.Holistic(static_image_mode=False, model_complexity=1, enable_segmentation=False, refine_face_landmarks=False, min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                result = holistic.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                body, bc = body_points(result.pose_landmarks, width, height, 33)
                left, lc = component_points(result.left_hand_landmarks, width, height, 21)
                right, rc = component_points(result.right_hand_landmarks, width, height, 21)
                data.append(np.concatenate((body, left, right), axis=0).astype(np.float32))
                confidence.append(np.concatenate((bc, lc, rc), axis=0).astype(np.float32))
    finally:
        cap.release()
    if not data:
        raise RuntimeError(f"Video không có frame: {video_path}")
    body = NumPyPoseBody(fps=fps, data=np.ma.masked_array(np.expand_dims(np.stack(data), 1)), confidence=np.expand_dims(np.stack(confidence), 1))
    pose = Pose(header=header, body=body)
    pose_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = pose_path.with_suffix(".pose.part")
    with temporary.open("wb") as f:
        pose.write(f)
    temporary.replace(pose_path)
    print(f"Đã trích pose {video_path.name}: {len(data)} frame.")

def prepare_features(split, frame):
    video_dir = SPLITS[split][1]
    pose_dir, feature_dir = POSE_ROOT / split, FEATURE_ROOT / split
    pose_dir.mkdir(parents=True, exist_ok=True)
    feature_dir.mkdir(parents=True, exist_ok=True)
    for index, name in enumerate(sorted(set(frame.VIDEO_NAME)), 1):
        feature_path = feature_dir / f"{name}.npz"
        if feature_path.is_file() and feature_path.stat().st_size:
            continue
        pose_path = pose_dir / f"{name}.pose"
        make_pose(video_dir / f"{name}.mp4", pose_path)
        pose = Pose.read(pose_path.read_bytes())
        normalized = preprocess_pose(pose)
        xyz = normalized.body.data.filled(0)[:, 0, :, :3].astype(np.float32)
        if xyz.shape[1:] != (50, 3):
            raise ValueError(f"Pose phải có dạng (T,50,3), nhận {xyz.shape}")
        fps = float(pose.body.fps)
        times = np.arange(len(xyz), dtype=np.float32) / fps
        features = np.concatenate((xyz, compute_velocity(xyz, times)), axis=-1).astype(np.float32)
        if features.shape[1:] != (50, 6):
            raise ValueError(f"Feature phải có dạng (T,50,6), nhận {features.shape}")
        np.savez_compressed(feature_path, features=features, timestamps=times, fps=np.asarray(fps, dtype=np.float32))
        print(f"{split}: cache {index}/{frame.VIDEO_NAME.nunique()} · {name}")

prepare_features("train", train_df)
prepare_features("validation", val_df)

In [ ]:
MAX_FRAMES = 1024

class How2SignBoundaryDataset(Dataset):
    def __init__(self, frame, split, training):
        self.feature_dir = FEATURE_ROOT / split
        self.training = training
        self.items = []
        for name, rows in frame.groupby("VIDEO_NAME", sort=True):
            self.items.append({
                "name": str(name),
                "feature_path": self.feature_dir / f"{name}.npz",
                "spans": [(float(r.START_REALIGNED), float(r.END_REALIGNED)) for r in rows.itertuples()],
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        item = self.items[index]
        with np.load(item["feature_path"]) as cached:
            features, times, fps = cached["features"], cached["timestamps"], float(cached["fps"])
        start, stop = 0, len(features)
        if self.training and stop > MAX_FRAMES:
            start = random.randint(0, stop - MAX_FRAMES)
            stop = start + MAX_FRAMES
        features = np.ascontiguousarray(features[start:stop])
        times = times[start:stop]
        t0, clip_end = float(times[0]), float(times[-1] + 1.0 / fps)
        spans_ms, crosses_start = [], False
        for begin, end in item["spans"]:
            if end <= t0 or begin >= clip_end:
                continue
            crosses_start |= begin < t0 < end
            spans_ms.append({"start": max(0.0, begin - t0) * 1000, "end": max(0.0, min(clip_end, end) - t0) * 1000})
        labels = create_bio_from_times(spans_ms, (times - t0) * 1000)
        if crosses_start and len(labels):
            labels[0] = BIO["I"]
        return {
            "pose": torch.from_numpy(features),
            "timestamps": torch.from_numpy(np.ascontiguousarray(times - t0)),
            "labels": torch.from_numpy(labels.astype(np.int64)),
        }

train_set = How2SignBoundaryDataset(train_df, "train", True)
val_set = How2SignBoundaryDataset(val_df, "validation", False)
train_loader = DataLoader(train_set, batch_size=1, shuffle=True, num_workers=0)
print(f"Train {len(train_set)} video · validation {len(val_set)} video · batch 1 · {MAX_FRAMES} frame/crop")

In [ ]:
IOU_THRESHOLDS = (0.1, 0.3, 0.5)

def interval_iou(a, b):
    intersection = max(0.0, min(a["end"], b["end"]) - max(a["start"], b["start"]))
    union = max(a["end"], b["end"]) - min(a["start"], b["start"])
    return intersection / union if union > 0 else 0.0

def greedy_matches(predicted, reference, threshold):
    pairs = sorted(((interval_iou(p, g), i, j) for i, p in enumerate(predicted) for j, g in enumerate(reference)), reverse=True)
    used_p, used_g, count = set(), set(), 0
    for overlap, i, j in pairs:
        if overlap < threshold:
            break
        if i not in used_p and j not in used_g:
            used_p.add(i)
            used_g.add(j)
            count += 1
    return count

@torch.inference_mode()
def evaluate(model, dataset):
    model.eval()
    counts = {t: [0, 0, 0] for t in IOU_THRESHOLDS}
    for item in dataset.items:
        with np.load(item["feature_path"]) as cached:
            x, times, fps = cached["features"], cached["timestamps"], float(cached["fps"])
        output = model(torch.from_numpy(x).unsqueeze(0).to(DEVICE), timestamps=torch.from_numpy(times).unsqueeze(0).to(DEVICE))["sentence"][0].float().cpu()
        segments = filter_segments(likeliest_probs_to_segments(output), min_frames=3, merge_gap=0)
        predicted = [{"start": s["start"] / fps, "end": (s["end"] + 1) / fps} for s in segments]
        reference = [{"start": a, "end": b} for a, b in item["spans"]]
        for t in IOU_THRESHOLDS:
            counts[t][0] += greedy_matches(predicted, reference, t)
            counts[t][1] += len(predicted)
            counts[t][2] += len(reference)
    result = {}
    for t, (matched, predicted_count, reference_count) in counts.items():
        precision = matched / predicted_count if predicted_count else 0.0
        recall = matched / reference_count if reference_count else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        result[str(t)] = {"precision": precision, "recall": recall, "f1": f1, "matched": matched, "predicted": predicted_count, "reference": reference_count}
    return result

baseline = load_model(str(BASE), device="cpu").to(DEVICE)
baseline_metrics = evaluate(baseline, val_set)
del baseline
torch.cuda.empty_cache()
print("Baseline DGS trên validation:", json.dumps(baseline_metrics, ensure_ascii=False, indent=2))

In [ ]:
import torch.nn.functional as F

EPOCHS, PATIENCE = 12, 4
BACKBONE_LR, SENTENCE_HEAD_LR = 3e-6, 3e-5

model = load_model(str(BASE), device="cpu", config_overrides={
    "pose_dims": (50, 6),
    "learning_rate": SENTENCE_HEAD_LR,
    "steps_per_epoch": len(train_loader),
    "max_epochs": EPOCHS,
    "num_frames": MAX_FRAMES,
}, eval_mode=False).to(DEVICE)

# How2Sign CSV không có nhãn gloss ASL; chỉ tối ưu sentence head và encoder.
for parameter in model.sign_bio_head.parameters():
    parameter.requires_grad = False
head = list(model.sentence_bio_head.parameters())
backbone = [p for n, p in model.named_parameters() if not n.startswith(("sentence_bio_head.", "sign_bio_head.")) and p.requires_grad]
optimizer = torch.optim.AdamW([{"params": backbone, "lr": BACKBONE_LR}, {"params": head, "lr": SENTENCE_HEAD_LR}], weight_decay=1e-2)
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=amp_dtype == torch.float16)
best_f1, best_state, best_metrics, bad_epochs = -1.0, None, None, 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for batch in train_loader:
        x, times, labels = batch["pose"].to(DEVICE), batch["timestamps"].to(DEVICE), batch["labels"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            log_probs = model(x, timestamps=times)["sentence"]
            loss = F.nll_loss(log_probs.transpose(1, 2), labels, ignore_index=BIO["UNK"])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(loss.detach().cpu()))

    metrics = evaluate(model, val_set)
    score = metrics["0.3"]["f1"]
    history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "validation": metrics})
    print(f"Epoch {epoch}/{EPOCHS} · loss {np.mean(losses):.4f} · val segment F1@IoU0.3 {score:.4f}")
    if score > best_f1:
        best_f1, best_state, best_metrics, bad_epochs = score, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, metrics, 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("Dừng sớm vì validation không cải thiện.")
            break

if best_state is None:
    raise RuntimeError("Không lưu được checkpoint tốt nhất.")
model.load_state_dict(best_state)
model.eval()
print("Metric baseline:", json.dumps(baseline_metrics, ensure_ascii=False))
print("Metric fine-tune tốt nhất:", json.dumps(best_metrics, ensure_ascii=False))

In [ ]:
from safetensors.torch import save_file

OUT.mkdir(parents=True, exist_ok=True)
save_file({k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()}, str(OUT / "model.safetensors"))
config = json.loads((BASE / "config.json").read_text())
config["pose_dims"] = [50, 6]
config["num_frames"] = MAX_FRAMES
(OUT / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
(OUT / "training_history.json").write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")
manifest = {
    "train_video_ids": sorted(train_ids),
    "validation_video_ids": sorted(val_ids),
    "test_split_used_for_training_or_tuning": False,
    "baseline_validation": baseline_metrics,
    "best_finetuned_validation": best_metrics,
    "selection_metric": "one-to-one segment F1 at temporal IoU 0.3",
    "annotation_note": "English translation timestamps approximate ASL sentence boundaries.",
}
(OUT / "training_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
reloaded = load_model(str(OUT), device="cpu")
print(f"Đã lưu và strict-load lại model: {OUT}")
print("Chưa chạy test split; giữ test riêng cho báo cáo cuối.")

## Cách kết luận

So sánh baseline và fine-tuned trên cùng validation video bằng segment precision/recall/F1 tại IoU 0,1 / 0,3 / 0,5. Không dùng frame F1 làm metric chính. Chưa kết luận được trên ASL nếu chưa cải thiện ở test held-out.

Đầu ra gồm model.safetensors, config.json, training_history.json và training_manifest.json trong thư mục checkpoints trên Google Drive. Checkpoint này chỉ phân đoạn câu; sau đó mới đưa các đoạn vào Uni-Sign để kiểm tra tác động lên dịch.